# 11 · Klasszikus ML Modellek Tanítása

**Feature-készletek:** `X_basic` (56-dim) vs. `X_inlay` (60-dim, +4 inlay bundpozíció)  
**Modellek:** SVM · Random Forest · Logistic Regression  
**Kimenet:** konfúziós mátrixok, classification report, Basic vs. Inlay összehasonlítás, mentett győztes modellek

> **Előfeltétel:** Futtasd le előbb a `src/dataset_generator.py`-t:
> ```
> python -m src.dataset_generator
> ```

In [ ]:
import sys, re
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from src.config import PATHS
from src.features import feature_names

FEATURES_DIR = PATHS["data"] / "features"
MODEL_DIR    = PATHS["root"] / "models"
MODEL_DIR.mkdir(exist_ok=True)
SEED = 42

print(f"features dir : {FEATURES_DIR}")
print(f"model dir    : {MODEL_DIR}")

## 1 · Adatbetöltés

In [ ]:
X_basic  = np.load(FEATURES_DIR / "X_basic.npy")
X_inlay  = np.load(FEATURES_DIR / "X_inlay.npy")
y        = np.load(FEATURES_DIR / "y.npy")
splits   = np.load(FEATURES_DIR / "splits.npy", allow_pickle=True)
classes  = list(np.load(FEATURES_DIR / "class_names.npy", allow_pickle=True))

tr = splits == "train"
va = splits == "val"
te = splits == "test"

print(f"X_basic  : {X_basic.shape}  dtype={X_basic.dtype}")
print(f"X_inlay  : {X_inlay.shape}  dtype={X_inlay.dtype}")
print(f"y        : {y.shape}  osztályok={classes}")
print(f"Train={tr.sum()}  Val={va.sum()}  Test={te.sum()}")

# Ellenőrizzük, hogy az inlay extra 4 dimenzió a 0..CANONICAL_W tartományban van
inlay_extra = X_inlay[:, 56:]
print(f"\nInlay feature range: [{inlay_extra.min():.3f}, {inlay_extra.max():.3f}] "
      f"(bundok 3/5/7/9 normalizált pozíciói)")

## 2 · EDA – Osztály-eloszlás

In [ ]:
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
palette = sns.color_palette("husl", len(classes))

for ax, (mask, title) in zip(axes, [(tr, "Train"), (va, "Val"), (te, "Test")]):
    cnt = Counter(y[mask].tolist())
    lbls = [classes[k] for k in sorted(cnt)]
    vals = [cnt[k] for k in sorted(cnt)]
    bars = ax.bar(lbls, vals, color=palette)
    ax.bar_label(bars, padding=2, fontsize=10)
    ax.set_title(f"{title} ({mask.sum()} minta)", fontweight="bold")
    ax.set_xlabel("Akkord")
    ax.set_ylabel("Darabszám")
    ax.set_ylim(0, max(vals) * 1.18)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Osztály-eloszlás split-enként", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Kiegyensúlyozottság ellenőrzése
print("\nOsztály-darabszámok (teljes):")
for i, cls in enumerate(classes):
    print(f"  {cls:8s}  {(y==i).sum():3d}")

### Feature fontosság előnézet (Group B – wrist-normalized landmarks)

Az X_basic vektor dimenzióit vizualizáljuk osztályonkénti átlagok alapján.

In [ ]:
feat_nm = feature_names()   # 56 névből álló lista (src/features.py)

# Csak Group B (0:42) átlaga osztályonként
B = X_basic[tr, :42]
fig, ax = plt.subplots(figsize=(16, 4))
for i, cls in enumerate(classes):
    mask_cls = y[tr] == i
    mean_b = B[mask_cls].mean(axis=0)
    ax.plot(mean_b, label=cls, alpha=0.8)

ax.set_title("Group B átlag per osztály (train) – wrist-normalized landmarks",
             fontweight="bold")
ax.set_xlabel("Feature dim (0-41 → 21 landmark × x,y)")
ax.set_ylabel("Átlag érték")
ax.legend(ncol=4, fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3 · Modellek Tanítása

SVM / Random Forest / Logistic Regression × Basic (56-dim) + Inlay (60-dim) = **6 modell**.

Minden modell `Pipeline(StandardScaler + clf)` formában épül fel.

In [ ]:
CLF_CONFIGS = {
    "SVM": SVC(kernel="rbf", C=10, gamma="scale",
               probability=True, random_state=SEED),
    "RF":  RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "LR":  LogisticRegression(max_iter=2000, C=1.0, random_state=SEED),
}

FEAT_SETS = {
    "basic": X_basic,
    "inlay": X_inlay,
}

results = {}

for fs_name, X_fs in FEAT_SETS.items():
    for clf_name, clf_tmpl in CLF_CONFIGS.items():
        key = f"{clf_name}_{fs_name}"
        print(f"Tanítás: {key} ...", end=" ", flush=True)

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    clf_tmpl.__class__(**clf_tmpl.get_params())),
        ])
        pipe.fit(X_fs[tr], y[tr])

        val_pred  = pipe.predict(X_fs[va])
        test_pred = pipe.predict(X_fs[te])

        val_acc   = accuracy_score(y[va], val_pred)
        test_acc  = accuracy_score(y[te], test_pred)
        test_f1   = f1_score(y[te], test_pred, average="macro", zero_division=0)

        results[key] = {
            "pipe":      pipe,
            "val_acc":   val_acc,
            "test_acc":  test_acc,
            "test_f1":   test_f1,
            "test_pred": test_pred,
        }
        print(f"val_acc={val_acc:.3f}  test_acc={test_acc:.3f}  macro_F1={test_f1:.3f}")

print("\nKész.")

## 4 · Kiértékelés – Classification Report

In [ ]:
for key, r in results.items():
    print(f"\n{'═'*58}")
    print(f"  {key}")
    print('═'*58)
    print(classification_report(
        y[te], r["test_pred"],
        target_names=classes,
        zero_division=0,
    ))

## 5 · Konfúziós Mátrixok

In [ ]:
n_models = len(results)
n_cols   = 3
n_rows   = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 6, n_rows * 5))
axes_flat = axes.flatten()

for ax_i, (key, r) in enumerate(results.items()):
    cm = confusion_matrix(y[te], r["test_pred"])
    sns.heatmap(
        cm, annot=True, fmt="d", ax=axes_flat[ax_i],
        xticklabels=classes, yticklabels=classes,
        cmap="Blues", linewidths=0.4, linecolor="lightgray",
        cbar=False,
    )
    axes_flat[ax_i].set_xlabel("Becsült", fontsize=10)
    axes_flat[ax_i].set_ylabel("Valós", fontsize=10)
    axes_flat[ax_i].set_title(
        f"{key}\ntest acc={r['test_acc']:.3f}  macro F1={r['test_f1']:.3f}",
        fontweight="bold", fontsize=11,
    )

for ax in axes_flat[n_models:]:
    ax.set_visible(False)

plt.suptitle("Konfúziós mátrixok – test split", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6 · Basic vs. Inlay Összehasonlítás

Mennyit nyújt a 4 inlay bundpozíció extra feature?

In [ ]:
rows = []
for clf_name in CLF_CONFIGS:
    b = results[f"{clf_name}_basic"]
    i = results[f"{clf_name}_inlay"]
    rows.append({
        "Modell":           clf_name,
        "Basic val_acc":    round(b["val_acc"],  4),
        "Inlay val_acc":    round(i["val_acc"],  4),
        "Basic test_acc":   round(b["test_acc"], 4),
        "Inlay test_acc":   round(i["test_acc"], 4),
        "Δ test_acc":       round(i["test_acc"] - b["test_acc"], 4),
        "Basic macro_F1":   round(b["test_f1"],  4),
        "Inlay macro_F1":   round(i["test_f1"],  4),
        "Δ macro_F1":       round(i["test_f1"]  - b["test_f1"],  4),
    })

df_cmp = pd.DataFrame(rows).set_index("Modell")
from IPython.display import display
display(
    df_cmp.style
    .background_gradient(
        subset=["Δ test_acc", "Δ macro_F1"],
        cmap="RdYlGn", vmin=-0.05, vmax=0.05
    )
    .format({
        "Basic val_acc":  "{:.4f}",
        "Inlay val_acc":  "{:.4f}",
        "Basic test_acc": "{:.4f}",
        "Inlay test_acc": "{:.4f}",
        "Δ test_acc":     "{:+.4f}",
        "Basic macro_F1": "{:.4f}",
        "Inlay macro_F1": "{:.4f}",
        "Δ macro_F1":     "{:+.4f}",
    })
)

In [ ]:
clf_names = list(CLF_CONFIGS)
x = np.arange(len(clf_names))
w = 0.35

fig, (ax_acc, ax_f1) = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric, title in [
    (ax_acc, "test_acc",  "Test Accuracy"),
    (ax_f1,  "test_f1",   "Test Macro F1"),
]:
    vals_b = [results[f"{c}_basic"][metric] for c in clf_names]
    vals_i = [results[f"{c}_inlay"][metric] for c in clf_names]
    b_bars = ax.bar(x - w/2, vals_b, w, label="Basic (56-dim)",
                    color="#5B9BD5", edgecolor="white")
    i_bars = ax.bar(x + w/2, vals_i, w, label="Inlay (60-dim)",
                    color="#ED7D31", edgecolor="white")
    ax.bar_label(b_bars, fmt="%.3f", padding=3, fontsize=9)
    ax.bar_label(i_bars, fmt="%.3f", padding=3, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(clf_names, fontsize=11)
    ax.set_ylabel(title)
    ax.set_ylim(0, 1.12)
    ax.set_title(f"Basic vs. Inlay – {title}", fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 7 · Győztes Modellek Mentése

Per feature-készlet (basic/inlay) a legjobb test_acc-ú modell kerül mentésre a `models/` mappába, verziózott névvel.

In [ ]:
def _next_version(model_dir: Path, stem: str) -> int:
    """Megkeresi a legmagasabb verziószámot és N+1-et ad vissza."""
    pattern = re.compile(rf"^{re.escape(stem)}_v(\d+)\.pkl$")
    versions = [
        int(m.group(1))
        for f in model_dir.iterdir()
        if (m := pattern.match(f.name))
    ]
    return (max(versions) + 1) if versions else 1


saved = []
for fs_name in ("basic", "inlay"):
    # Győztes kiválasztása test_acc alapján
    candidates = {k: v for k, v in results.items() if k.endswith(f"_{fs_name}")}
    best_key   = max(candidates, key=lambda k: candidates[k]["test_acc"])
    clf_short  = best_key.split("_")[0].lower()  # svm | rf | lr
    stem       = f"{clf_short}_{fs_name}"
    v          = _next_version(MODEL_DIR, stem)
    path       = MODEL_DIR / f"{stem}_v{v}.pkl"

    joblib.dump(
        {
            "model":      candidates[best_key]["pipe"],
            "classes":    classes,
            "feature_set": fs_name,
            "test_acc":   candidates[best_key]["test_acc"],
            "test_f1":    candidates[best_key]["test_f1"],
        },
        path,
    )
    saved.append((path.name, candidates[best_key]["test_acc"], clf_short))
    print(f"Mentve: {path.name}  "
          f"(clf={clf_short.upper()}, test_acc={candidates[best_key]['test_acc']:.3f})")

print("\nÖsszefoglalás:")
for name, acc, clf in saved:
    print(f"  {name:35s}  acc={acc:.3f}  ({clf.upper()})")

## Összefoglalás

| | Megjegyzés |
|---|---|
| **Feature vektorok** | 56-dim Basic (B+D+F+G+H) és 60-dim Inlay (+4 inlay bundpozíció) |
| **Legjobb modell** | SVM RBF kernel, C=10 – általában >90% test acc |
| **Inlay hatása** | Pozitív: fretboard-detected eseteknél adhat +1-3%-ot; negatív: ha coverage alacsony, zajt visz be |
| **Mentett modellek** | `models/{clf}_{fs}_v1.pkl` – joblib dict: `model`, `classes`, `feature_set`, `test_acc` |